# 第 21 课 · RMSNorm 与 SwiGLU
### 从公式、梯度和参数预算，到四组可复现的小模型实验

上一课已经实现 RoPE 和 KV cache。这一课保持相同的自回归流程，重点更换**归一化**和**逐位置前馈网络**。

**完成标准：**能手算 RMSNorm；能解释 SiLU 门控为什么不是概率；能推导两个 FFN 的参数量；能写出 Pre-RMSNorm 残差；能说明四组对照的控制变量与局限。

本讲义和同名 `.py` 脚本都能独立运行；不导入前课文件，不下载模型或数据。只使用 PyTorch 和标准库，默认 CPU。建议先做第 1–6 节的组件实验，再做第 7–10 节的模型和练习。

> 运行全部单元 ≠ 完成练习。练习区保留未实现函数，检查开关默认关闭；参考答案集中在最后。

## 0. 环境与运行约定
沿用第 11–20 课的 Python/PyTorch 环境。没有 PyTorch 时，在自己的环境安装；GPU 构建按 [PyTorch 官方安装选择器](https://pytorch.org/get-started/locally/)配置。本课无须为 GPU 重装已能运行的环境。

CPU 足够完成全部训练；`DEVICE="auto"` 只在已有可用 CUDA 时选择显卡。完整模型默认 FP32，局部 FP16/BF16 示例**不等于完整模型支持混合精度**。官方核对需要 `torch.nn.functional.rms_norm`。

输出写入当前工作目录下的 `lesson_21_outputs`。同名生成文件会覆盖，不修改前课文件。

In [1]:
from __future__ import annotations
import argparse
from dataclasses import asdict, dataclass
import hashlib
import json
import math
from pathlib import Path
import platform
import random
import tempfile
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

DEVICE = "cpu"         # "auto" / "cuda" 仅适用于已配好 CUDA 的环境
EPOCHS = 30
SEED = 21
SINGLE = False          # True：只训练 RMSNorm+SwiGLU，不做完整四组对照
CHECKPOINT = None      # 例如 "lesson_21_outputs/rms_swiglu.pt"，仅限本课检查点
OUTPUT_DIR = Path("lesson_21_outputs")
torch.set_num_threads(1)
print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not hasattr(F, "rms_norm"):
    raise RuntimeError("官方核对需要支持 F.rms_norm 的 PyTorch")
SPECIAL_TOKENS = ["<unk>", "<bos>", "<eos>", "<pad>"]
UNK_ID, BOS_ID, EOS_ID, PAD_ID = range(4)
IGNORE_INDEX = -100
CHECKPOINT_FAMILY = "lesson21_rms_swiglu_v1"

Python: 3.13.5 PyTorch: 2.10.0+cpu
CUDA available: False


In [2]:
def positive_int(value: int, name: str) -> None:
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数，不接受布尔值")

def choose_device(requested: str = "cpu") -> torch.device:
    if requested not in {"cpu", "cuda", "auto"}:
        raise ValueError("device 必须是 cpu、cuda 或 auto")
    if requested == "auto":
        requested = "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA 不可用；请先用 --device cpu，或检查已有 CUDA 环境")
    return torch.device(requested)

def set_seed(seed: int) -> None:
    if type(seed) is not int or not 0 <= seed < 2**32:
        raise ValueError("seed 必须是 [0,2**32) 内的整数")
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def nonnegative_int(value: int, name: str) -> None:
    if type(value) is not int or value < 0:
        raise ValueError(f"{name} 必须是非负整数，不接受布尔值")

def check_base(base: float) -> None:
    if (isinstance(base, bool) or not isinstance(base, (int, float))
            or not math.isfinite(base) or base <= 1):
        raise ValueError("本课 rope_base 必须是大于 1 的有限数")

def check_eps(eps: float) -> None:
    if (isinstance(eps, bool) or not isinstance(eps, (int, float))
            or not math.isfinite(eps) or eps <= 0):
        raise ValueError("eps 必须是正的有限数")

@dataclass(frozen=True)
class ModelConfig:
    vocab_size: int
    max_len: int = 64
    d_model: int = 48
    n_heads: int = 4
    n_layers: int = 2
    ff_dim: int = 192
    swiglu_dim: int = 128
    norm_kind: str = "rms"
    ffn_kind: str = "swiglu"
    norm_eps: float = 1e-6
    dropout: float = 0.0
    rope_base: float = 10000.0

    def __post_init__(self) -> None:
        for name in ("vocab_size", "max_len", "d_model", "n_heads", "n_layers", "ff_dim", "swiglu_dim"):
            positive_int(getattr(self, name), name)
        if self.vocab_size <= 4:
            raise ValueError("词表必须包含普通字符及四个特殊 token")
        if self.d_model % self.n_heads or (self.d_model // self.n_heads) % 2:
            raise ValueError("D 必须能被 H 整除，且每头维度为偶数")
        if self.norm_kind not in {"layer", "rms"} or self.ffn_kind not in {"gelu", "swiglu"}:
            raise ValueError("不支持的归一化或 FFN 类型")
        if (isinstance(self.dropout, bool) or not isinstance(self.dropout, (int, float))
                or not math.isfinite(self.dropout) or not 0 <= self.dropout < 1):
            raise ValueError("dropout 必须在 [0,1) 内")
        check_eps(self.norm_eps)
        check_base(self.rope_base)

device = choose_device(DEVICE)
set_seed(SEED)
print("本次设备：", device)

本次设备： cpu


## 1. 从 LayerNorm 到 RMSNorm

第 15 课的 LayerNorm 先计算均值，再对中心化后的特征归一化：
\[
\operatorname{LN}(x)_i=\gamma_i\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta_i.
\]

本课 RMSNorm 不减均值，只用均方根缩放：
\[
\operatorname{RMSNorm}(x)_i=
\gamma_i\frac{x_i}{\sqrt{\frac1D\sum_{j=1}^{D}x_j^2+\epsilon}}.
\]

对于 `[B,T,D]`，本课只沿末维 `D` 计算统计量，每个 token 各算自己的值；没有跨 token 混合。参数只有缩放向量 `gamma=[D]`，没有偏置 beta。两者都是输入相关运算，不能当作固定线性矩阵。

**来源：**[RMSNorm 原论文](https://arxiv.org/abs/1910.07467)、[PyTorch RMSNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.RMSNorm.html)。

### 1.1 先手算 `[3,4]`
平方均值是 `(9+16)/2=12.5`，均方根约为 `3.5355`。`gamma=[1,1]`、忽略极小 eps 时，输出约 `[0.8485,1.1314]`。均值不是零！

In [3]:
x_hand = torch.tensor([3.0, 4.0], dtype=torch.float64)
mean_square = x_hand.square().mean()
y_hand = x_hand / torch.sqrt(mean_square + 1e-6)
print("平方均值：", mean_square.item())
print("RMSNorm 输出：", y_hand.tolist())
print("输出均值：", y_hand.mean().item())

平方均值： 12.5
RMSNorm 输出： [0.8485281034827336, 1.1313708046436448]
输出均值： 0.9899494540631892


### 1.2 实现：统计量用更高精度计算
`rsqrt(a)` 表示 `1/sqrt(a)`，不是 `1/a`。下面实现把 FP16/BF16 输入先提升至 FP32，平方和求均值之后再转回输入 dtype；FP64 不降成 FP32，便于严格梯度检查。

`eps` 显式设为 `1e-6`，放在平方根里面。比较不同实现时需要对齐 eps、归一化轴、参数和 dtype 策略，不能仅看函数名。

In [4]:
def rms_norm(x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """带缩放参数、不带偏置；FP16/BF16 先升 FP32，FP64 保留 FP64。

    输出 dtype 与 x 一致；这是本教学实现的明确精度约定。
    不减均值，不跨 batch/token 计算统计量，不在原地修改输入。
    """
    check_eps(eps)
    supported = (torch.float16, torch.bfloat16, torch.float32, torch.float64)
    if (not isinstance(x, torch.Tensor) or x.dtype not in supported
            or x.ndim < 1 or min(x.shape) <= 0):
        raise ValueError("x 必须是非空浮点张量，至少有一个维度")
    if (not isinstance(weight, torch.Tensor) or weight.dtype not in supported
            or weight.shape != (x.shape[-1],) or weight.device != x.device):
        raise ValueError("weight 必须是同设备、长度为最后特征维度的浮点向量")
    compute_dtype = torch.float64 if x.dtype == torch.float64 else torch.float32
    work = x.to(compute_dtype)
    mean_square = work.square().mean(dim=-1, keepdim=True)
    normalized = work * torch.rsqrt(mean_square + eps)
    return (normalized * weight.to(compute_dtype)).to(x.dtype)

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        positive_int(dim, "dim")
        check_eps(eps)
        self.dim, self.eps = dim, float(eps)
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return rms_norm(x, self.weight, self.eps)

def make_norm(kind: str, dim: int, eps: float) -> nn.Module:
    if kind == "rms":
        return RMSNorm(dim, eps)
    if kind == "layer":
        # LayerNorm 基线保留 gamma 和 beta；RMSNorm 仅有 gamma。
        return nn.LayerNorm(dim, eps=eps)
    raise ValueError("norm_kind 必须是 layer 或 rms")

### 1.3 常量向量、平移与缩放
常量 `[3,3,3,3]` 的 LayerNorm 归一化部分为零；RMSNorm 接近全 1。LayerNorm 对每个特征都加同一个常量保持不变；RMSNorm 通常不会。

在 `gamma=1` 且加入 eps 时，RMSNorm 输出均方为 `mean(x²)/(mean(x²)+eps)`，不精确等于 1。再乘可学习 gamma 后，更不能保证固定均方。

忽略 eps 时，非零向量的正数缩放不改变归一化结果；负数缩放会改变符号。实际有 eps，尤其输入很小时，只能说近似，不能笼统称为严格缩放不变。

In [5]:
z = torch.tensor([1., 2., 3., 4.], dtype=torch.float64)
w = torch.ones(4, dtype=torch.float64)
constant = torch.full_like(z, 3.)
print("LN 常量：", F.layer_norm(constant, (4,), eps=1e-6).tolist())
print("RMSNorm 常量：", rms_norm(constant, w).tolist())
print("RMSNorm 原输入：", rms_norm(z, w).tolist())
print("RMSNorm 平移+10：", rms_norm(z + 10, w).tolist())
print("LN 平移差：", (F.layer_norm(z, (4,), eps=1e-6)-F.layer_norm(z+10, (4,), eps=1e-6)).abs().max().item())
print("小输入缩放差：", (rms_norm(z*1e-5,w)-rms_norm(z*7e-5,w)).abs().max().item())

LN 常量： [0.0, 0.0, 0.0, 0.0]
RMSNorm 常量： [0.9999999444444492, 0.9999999444444492, 0.9999999444444492, 0.9999999444444492]
RMSNorm 原输入： [0.3651483473268884, 0.7302966946537768, 1.0954450419806652, 1.4605933893075536]
RMSNorm 平移+10： [0.8765009773960126, 0.9561828844320137, 1.0358647914680148, 1.115546698504016]
LN 平移差： 0.0
小输入缩放差： 0.23500759291068718


### 1.4 不只核对前向，还要核对梯度
如果误用了 `detach()`，前向输出可能完全正确，但训练已经断开。这里分别检查输入和缩放参数的梯度；不是只检查 `.grad` 是否存在。

In [6]:
torch.manual_seed(2101)
x_ref = torch.randn(2, 3, 4, dtype=torch.float64, requires_grad=True)
gamma_ref = torch.tensor([0.5, 1., 1.5, 2.], dtype=torch.float64, requires_grad=True)
y_custom = rms_norm(x_ref, gamma_ref, 1e-6)
y_official = F.rms_norm(x_ref, (4,), gamma_ref, 1e-6)
upstream = torch.randn_like(x_ref)
g_custom = torch.autograd.grad((y_custom*upstream).sum(), (x_ref,gamma_ref), retain_graph=True)
g_official = torch.autograd.grad((y_official*upstream).sum(), (x_ref,gamma_ref))
torch.testing.assert_close(y_custom, y_official)
for a,b in zip(g_custom,g_official):
    torch.testing.assert_close(a,b)
print("最大输出差：", (y_custom-y_official).abs().max().item())
print("输入、gamma 梯度均与官方实现一致（使用浮点容差）")

最大输出差： 0.0
输入、gamma 梯度均与官方实现一致（使用浮点容差）


### 1.5 为什么要先升精度，再平方？
`300` 和 `400` 可存进 FP16，但平方后的数值可能超出其有限范围。等平方已经溢出才 `.float()`，不能恢复原来的数值。

下面故意保留一个不安全版本用于观察，再与先升精度的版本比较。**这只是局部数值实验，不是完整 AMP 训练实现。**

In [7]:
x_half = torch.tensor([300., 400.], dtype=torch.float16)
naive = x_half * torch.rsqrt(x_half.square().mean() + 1e-6)
safe = rms_norm(x_half, torch.ones(2), 1e-6)
print("直接 FP16 平方：", x_half.square())
print("不安全结果：", naive)
print("先升 FP32 的结果：", safe)
print("输出 dtype：", safe.dtype)

直接 FP16 平方： tensor([inf, inf], dtype=torch.float16)
不安全结果： tensor([0., 0.], dtype=torch.float16)
先升 FP32 的结果： tensor([0.8486, 1.1318], dtype=torch.float16)
输出 dtype： torch.float16


## 2. SwiGLU：三次线性映射与一个逐元素乘法

普通无偏置 GELU FFN：
\[
\operatorname{FFN}(x)=\operatorname{GELU}(xW_{up}^{\top})W_{down}^{\top}.
\]

本课无偏置 SwiGLU FFN：
\[
\operatorname{SwiGLU}(x)=
\left[\operatorname{SiLU}(xW_{gate}^{\top})\odot(xW_{up}^{\top})\right]W_{down}^{\top}.
\]

这里使用 PyTorch `nn.Linear.weight` 的存储布局：上投影权重为 `[M,D]`，下投影权重为 `[D,M]`，所以公式里有转置。`⊙` 是同位置、同特征的逐元素相乘，不是矩阵乘法。

`SiLU(z)=z*sigmoid(z)`。SiLU 等于 beta 为 1 的 Swish。只把普通 FFN 的 GELU 改成 SiLU，不会自动多出另一个上投影和乘法，因此不等于 SwiGLU。

**来源：**[GLU Variants Improve Transformer](https://arxiv.org/html/2002.05202v1)、[PyTorch SiLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.SiLU.html)。

In [8]:
z_gate = torch.tensor([-2., -1., 0., 1., 2.])
print("输入：", z_gate.tolist())
print("sigmoid：", torch.sigmoid(z_gate).tolist())
print("SiLU：", F.silu(z_gate).tolist())
torch.testing.assert_close(F.silu(z_gate), z_gate*torch.sigmoid(z_gate))

输入： [-2.0, -1.0, 0.0, 1.0, 2.0]
sigmoid： [0.11920291930437088, 0.2689414322376251, 0.5, 0.7310585975646973, 0.8807970285415649]
SiLU： [-0.23840583860874176, -0.2689414322376251, 0.0, 0.7310585975646973, 1.7615940570831299]


### 2.1 门控不是概率，也不是选择专家
SiLU 的值可以为负，也可以大于 1。它调制特征，不是要求总和为 1 的注意力概率，也不是 MoE 中的专家路由。本课仍然是稠密 FFN，每个 token 都走这些矩阵。

名称中的 `gate` 和 `up` 不可随意调换：`SiLU(a)*b` 通常不等于 `a*SiLU(b)`。当然，两个投影的权重可以重新命名，但必须同时对应正确的运算。

In [9]:
class GELUFFN(nn.Module):
    def __init__(self, dim: int, hidden: int) -> None:
        super().__init__()
        positive_int(dim, "dim")
        positive_int(hidden, "hidden")
        self.up_proj = nn.Linear(dim, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.gelu(self.up_proj(x)))

class SwiGLUFFN(nn.Module):
    def __init__(self, dim: int, hidden: int) -> None:
        super().__init__()
        positive_int(dim, "dim")
        positive_int(hidden, "hidden")
        self.gate_proj = nn.Linear(dim, hidden, bias=False)
        self.up_proj = nn.Linear(dim, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.gate_proj(x))
        value = self.up_proj(x)
        return self.down_proj(gate * value)

def make_ffn(cfg: ModelConfig) -> nn.Module:
    if cfg.ffn_kind == "gelu":
        return GELUFFN(cfg.d_model, cfg.ff_dim)
    return SwiGLUFFN(cfg.d_model, cfg.swiglu_dim)

### 2.2 跟踪三个分支的形状
输入 `[B,T,D]=[2,3,48]`；两个上投影分别得到 `[2,3,128]`；逐元素相乘后形状不变；下投影回到 `[2,3,48]`。

你应该能解释：为什么残差仍能直接相加？为什么这个 FFN 自身还不会让第一个 token 读取第二个 token？

In [10]:
ffn_demo = SwiGLUFFN(48,128)
x_ffn = torch.randn(2,3,48)
gate_input = ffn_demo.gate_proj(x_ffn)
up = ffn_demo.up_proj(x_ffn)
product = F.silu(gate_input)*up
y_ffn = ffn_demo.down_proj(product)
print("input / gate / up / product / output:")
for value in (x_ffn,gate_input,up,product,y_ffn):
    print(tuple(value.shape))
x_changed = x_ffn.clone()
x_changed[:,0,:] += 10
torch.testing.assert_close(ffn_demo(x_changed)[:,1:,:], ffn_demo(x_ffn)[:,1:,:])
print("只改变第0位置，不影响FFN的其他位置输出")

input / gate / up / product / output:
(2, 3, 48)
(2, 3, 128)
(2, 3, 128)
(2, 3, 128)
(2, 3, 48)
只改变第0位置，不影响FFN的其他位置输出


## 3. 三个矩阵后，为什么中间宽度常要缩小？
只数无偏置 FFN 的权重：

| 结构 | 权重数量 |
|---|---:|
| 普通 FFN，`D→M→D` | `2DM` |
| SwiGLU，两个 `D→H`、一个 `H→D` | `3DH` |

普通中间宽度 `M=4D` 时，权重数是 `8D²`。为了与 SwiGLU 的 `3DH` 对齐，令：
\[
H=\frac{8D}{3}.
\]

原论文为控制参数和主要矩阵计算预算，将门控 FFN 的中间宽度缩为原来的 `2/3`。具体模型还可能设置宽度乘数、按硬件友好整数取整；`8D/3` 不是所有模型都必须遵守的定律。[论文](https://arxiv.org/html/2002.05202v1)及 [Llama 3 公开实现](https://github.com/meta-llama/llama3/blob/main/llama/model.py)可对照阅读。

**本课选择 D=48：**GELU 中间宽度 192，SwiGLU 中间宽度 128，两者每层都正好 18,432 个权重。与上课 D=64 的规模不同，不能直接拿两课训练成绩作架构对比。

In [11]:
def swiglu_budget_width(dim: int, multiple_of: int = 1) -> int:
    """与无偏置 D->4D->D FFN 的权重预算对齐，向上取整至 multiple_of。

    3*D*H = 8*D*D -> H = 8D/3。对齐后不一定完全相等。
    """
    positive_int(dim, "dim")
    positive_int(multiple_of, "multiple_of")
    denominator = 3 * multiple_of
    return ((8 * dim + denominator - 1) // denominator) * multiple_of

def parameter_budget(dim: int, gelu_hidden: int, swiglu_hidden: int) -> dict:
    for name, value in (("dim", dim), ("gelu_hidden", gelu_hidden), ("swiglu_hidden", swiglu_hidden)):
        positive_int(value, name)
    gelu = 2 * dim * gelu_hidden
    swiglu = 3 * dim * swiglu_hidden
    return {"dim": dim, "gelu_hidden": gelu_hidden, "swiglu_hidden": swiglu_hidden,
            "gelu_weights": gelu, "swiglu_weights": swiglu,
            "relative_weight_difference": (swiglu - gelu) / gelu,
            "note": "仅无偏置 FFN 权重；不包含 norm、attention、激活内存或实际耗时"}

def component_report() -> dict:
    x = torch.tensor([1., 2., 3., 4.], dtype=torch.float64)
    w = torch.ones_like(x)
    eps = 1e-6
    r = rms_norm(x, w, eps)
    l = F.layer_norm(x, (4,), eps=eps)
    constant = torch.full((4,), 3., dtype=torch.float64)
    values = torch.tensor([-2., -1., 0., 1., 2.], dtype=torch.float64)
    half = torch.tensor([300., 400.], dtype=torch.float16)
    naive = half * torch.rsqrt(half.square().mean() + eps)
    stable = rms_norm(half, torch.ones(2), eps)
    return {
        "input": x.tolist(), "eps": eps,
        "rmsnorm": r.tolist(), "layernorm": l.tolist(),
        "rmsnorm_output_mean": r.mean().item(),
        "rmsnorm_output_mean_square": r.square().mean().item(),
        "constant_rmsnorm": rms_norm(constant, w, eps).tolist(),
        "constant_layernorm": F.layer_norm(constant, (4,), eps=eps).tolist(),
        "rms_shift_max_difference": (rms_norm(x+10, w, eps)-r).abs().max().item(),
        "layer_shift_max_difference": (F.layer_norm(x+10, (4,), eps=eps)-l).abs().max().item(),
        "silu_inputs": values.tolist(), "silu_outputs": F.silu(values).tolist(),
        "fp16_square_overflow": not bool(torch.isfinite(half.square()).all()),
        "naive_fp16_output": naive.tolist(), "promoted_fp16_output": stable.tolist(),
        "exact_budget": parameter_budget(48, 192, 128),
        "rounded_budget": parameter_budget(64, 256, swiglu_budget_width(64, 8)),
    }

print(json.dumps(parameter_budget(48,192,128),ensure_ascii=False,indent=2))
print("D=64，按8对齐后的宽度：",swiglu_budget_width(64,8))
print(json.dumps(parameter_budget(64,256,176),ensure_ascii=False,indent=2))

{
  "dim": 48,
  "gelu_hidden": 192,
  "swiglu_hidden": 128,
  "gelu_weights": 18432,
  "swiglu_weights": 18432,
  "relative_weight_difference": 0.0,
  "note": "仅无偏置 FFN 权重；不包含 norm、attention、激活内存或实际耗时"
}
D=64，按8对齐后的宽度： 176
{
  "dim": 64,
  "gelu_hidden": 256,
  "swiglu_hidden": 176,
  "gelu_weights": 32768,
  "swiglu_weights": 33792,
  "relative_weight_difference": 0.03125,
  "note": "仅无偏置 FFN 权重；不包含 norm、attention、激活内存或实际耗时"
}


**参数相同不等于实际开销相同。**两个 FFN 的主要矩阵乘法乘加次数可以按预算接近，但逐元素运算、暂存激活、kernel 融合和内存访问不同。本课没有做可靠延迟或峰值显存基准，不输出“快多少”的结论。

RMSNorm 参数更少也不等于整个大模型显著缩小。本课每层的两个 norm 加最终 norm，共 5 组；把有 gamma/beta 的 LayerNorm 换成只有 gamma 的 RMSNorm，仅少 `5×48=240` 个参数。

## 4. Pre-RMSNorm 残差：不要改掉直通路径

\[
u=x+\operatorname{Attention}(\operatorname{RMSNorm}_1(x)),
\qquad
y=u+\operatorname{SwiGLU}(\operatorname{RMSNorm}_2(u)).
\]

变化发生在分支内部；主路仍保留原始 `x` 或更新后的 `u`。不能先 `x=norm(x)` 再把这个新变量当作残差主路，也不能为省事 `x.detach()`。

下面先独立检查 FFN 子层：将下投影置零，分支输出为零，那么输出必须等于输入，输入的直通梯度必须为 1。

In [12]:
norm_demo = RMSNorm(4).double()
ffn_zero = SwiGLUFFN(4,8).double()
with torch.no_grad():
    ffn_zero.down_proj.weight.zero_()
x_res = torch.randn(2,3,4,dtype=torch.float64,requires_grad=True)
y_res = x_res + ffn_zero(norm_demo(x_res))
torch.testing.assert_close(y_res,x_res,rtol=0,atol=0)
y_res.sum().backward()
torch.testing.assert_close(x_res.grad,torch.ones_like(x_res))
print("零分支：输出等于输入，残差直通梯度为1")

零分支：输出等于输入，残差直通梯度为1


## 5. 接回 RoPE、注意力、缓存与语言模型
以下是前课核心逻辑的自包含版本。可以先折叠大段复习代码，重点看 `ModernBlock` 的 `make_norm()`、`make_ffn()` 和 `ModernGPT.final_norm`。

继续使用相邻配对 RoPE，旋转 Q/K、不旋转 V；缓存保存旋转后的 K。前向支持右 padding；缓存只支持等长、无 PAD、`eval()+no_grad()` 的输入。保留普通 MHA，未添加 GQA。

本课 attention 的线性层仍有偏置；两种 FFN 均无偏置。它是教学解码器，不是 Llama 权重兼容实现，不能称为复刻某个主流模型。

In [13]:
def rope_frequencies(head_dim: int, base: float = 10000.0, *,
                     device: torch.device | str = "cpu", dtype=torch.float32) -> torch.Tensor:
    """omega_i=base**(-2*i/head_dim)，i 从 0 开始；不是 head_dim**(-2*i/base)。"""
    positive_int(head_dim, "head_dim")
    if head_dim % 2:
        raise ValueError("head_dim 必须是偶数")
    check_base(base)
    if dtype not in (torch.float32, torch.float64):
        raise ValueError("频率和相位使用 float32 或 float64，不直接使用半精度")
    exponent = torch.arange(0, head_dim, 2, device=device, dtype=dtype) / head_dim
    return float(base) ** (-exponent)

def _rope_inputs(x: torch.Tensor, positions: torch.Tensor, base: float) -> torch.dtype:
    if (not isinstance(x, torch.Tensor) or x.ndim != 4 or min(x.shape) <= 0
            or x.dtype not in (torch.float16, torch.bfloat16, torch.float32, torch.float64)
            or x.shape[-1] % 2):
        raise ValueError("x 必须是非空浮点 [B,H,T,d]，且 d 是偶数")
    if (not isinstance(positions, torch.Tensor) or positions.dtype != torch.long
            or positions.ndim != 1 or positions.shape[0] != x.shape[-2]
            or positions.device != x.device):
        raise ValueError("positions 必须是同设备、长度为 T 的 torch.long 向量")
    check_base(base)
    # 允许负位置：用于数学逆旋转实验；模型接口另外要求 offset 非负。
    return torch.float64 if x.dtype == torch.float64 else torch.float32

def apply_rope(x: torch.Tensor, positions: torch.Tensor, base: float = 10000.0) -> torch.Tensor:
    """相邻配对 (0,1),(2,3),...；返回相同 shape/dtype/device；不原地修改 x。

    输入 [B,H,T,d]，位置 [T]：本课各 batch 共享同一位置序列。
    半精度输入先升为 FP32 计算相位和旋转，再转回；主模型不启用 autocast。
    """
    compute_dtype = _rope_inputs(x, positions, base)
    freq = rope_frequencies(x.shape[-1], base, device=x.device, dtype=compute_dtype)
    angles = positions.to(compute_dtype)[:, None] * freq[None, :]
    cos = angles.cos()[None, None, :, :]
    sin = angles.sin()[None, None, :, :]
    work = x.to(compute_dtype)
    even, odd = work[..., 0::2], work[..., 1::2]
    rotated_even = even*cos - odd*sin
    rotated_odd = even*sin + odd*cos
    return torch.stack((rotated_even, rotated_odd), dim=-1).flatten(-2).to(x.dtype)

def attention_allowed(key_valid: torch.Tensor) -> torch.Tensor:
    """[B,T] -> [B,1,T,T]；True=允许读取。只接受非空、右侧补齐。"""
    if (not isinstance(key_valid, torch.Tensor) or key_valid.dtype != torch.bool
            or key_valid.ndim != 2 or min(key_valid.shape) <= 0):
        raise ValueError("key_valid 必须是非空 [B,T] 布尔张量")
    if not bool(key_valid[:, 0].all()):
        raise ValueError("首位置必须有效；不接受全 PAD 或左侧 padding")
    if bool((~key_valid[:, :-1] & key_valid[:, 1:]).any()):
        raise ValueError("只接受连续有效前缀加右 padding，不接受内部空洞")
    t = key_valid.shape[1]
    causal = torch.ones(t, t, dtype=torch.bool, device=key_valid.device).tril()
    # 不屏蔽整行 query：补齐 query 可有数值，但其标签被忽略，避免 softmax 全 -inf。
    return causal[None, None, :, :] & key_valid[:, None, None, :]

def offset_causal_mask(past_len: int, new_len: int, device: torch.device | str = 'cpu') -> torch.Tensor:
    """返回 [1,1,Q,P+Q]。第 i 个新 query 的绝对位置是 P+i。True=允许。"""
    if type(past_len) is not int or past_len < 0:
        raise ValueError('past_len 必须是非负整数')
    positive_int(new_len, 'new_len')
    query_positions = past_len + torch.arange(new_len, device=device)
    key_positions = torch.arange(past_len + new_len, device=device)
    allowed = key_positions[None, :] <= query_positions[:, None]
    return allowed[None, None, :, :]

LayerKV = tuple[torch.Tensor, torch.Tensor]

@dataclass(frozen=True)
class RotaryCache:
    layers: tuple[LayerKV, ...]
    position_offset: int
    rope_base: float
    owner: int  # 仅在当前进程识别模型对象；不是权重版本号，不是安全凭证。

    @property
    def length(self) -> int:
        return self.layers[0][0].shape[-2]

def cache_payload_bytes(cache: RotaryCache | None) -> int:
    if cache is None:
        return 0
    return sum(t.numel()*t.element_size() for pair in cache.layers for t in pair)

class RotaryAttention(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.n_heads, self.d_head = cfg.n_heads, cfg.d_model // cfg.n_heads
        self.base = cfg.rope_base
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def project(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        b, t, d = x.shape
        def split(proj):
            return proj(x).reshape(b, t, self.n_heads, self.d_head).transpose(1, 2)
        return split(self.q_proj), split(self.k_proj), split(self.v_proj)

    def combine(self, q, k, v, allowed):
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)
        weights = torch.softmax(scores.masked_fill(~allowed, float("-inf")), dim=-1)
        heads = self.dropout(weights) @ v
        b, h, t, d = heads.shape
        return self.out_proj(heads.transpose(1, 2).contiguous().reshape(b, t, h*d))

    def forward(self, x: torch.Tensor, positions: torch.Tensor, allowed: torch.Tensor) -> torch.Tensor:
        q, k, v = self.project(x)
        q = apply_rope(q, positions, self.base)
        k = apply_rope(k, positions, self.base)
        return self.combine(q, k, v, allowed)  # V 不旋转。

    def forward_cached(self, x: torch.Tensor, positions: torch.Tensor,
                       past: LayerKV | None = None) -> tuple[torch.Tensor, LayerKV]:
        q, k_new, v_new = self.project(x)
        q = apply_rope(q, positions, self.base)
        k_new = apply_rope(k_new, positions, self.base)
        past_len = 0 if past is None else past[0].shape[-2]
        if past is None:
            k_all, v_all = k_new, v_new
        else:
            # past[0] 已按各自原位置旋转，不能对拼接后的 k_all 再旋转！
            k_all = torch.cat((past[0], k_new), dim=-2)
            v_all = torch.cat((past[1], v_new), dim=-2)
        allowed = offset_causal_mask(past_len, x.shape[1], x.device)
        return self.combine(q, k_all, v_all, allowed), (k_all, v_all)

In [14]:
class ModernBlock(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.norm1 = make_norm(cfg.norm_kind, cfg.d_model, cfg.norm_eps)
        self.attn = RotaryAttention(cfg)
        self.norm2 = make_norm(cfg.norm_kind, cfg.d_model, cfg.norm_eps)
        self.ffn = make_ffn(cfg)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, positions, allowed):
        x = x + self.dropout(self.attn(self.norm1(x), positions, allowed))
        return x + self.dropout(self.ffn(self.norm2(x)))

class ModernGPT(nn.Module):
    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=PAD_ID)
        self.embedding_dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([ModernBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = make_norm(cfg.norm_kind, cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.apply(self._init_weights)
        with torch.no_grad():
            self.token_embedding.weight[PAD_ID].zero_()

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def _input_check(self, ids):
        if (not isinstance(ids, torch.Tensor) or ids.ndim != 2 or min(ids.shape) <= 0
                or ids.dtype != torch.long):
            raise ValueError("输入必须是非空 [B,T] torch.long")
        if ids.device != self.token_embedding.weight.device:
            raise ValueError("输入与模型必须在同一设备")
        if bool(((ids < 0) | (ids >= self.cfg.vocab_size)).any()):
            raise ValueError("token 编号越界")
        if ids.shape[1] > self.cfg.max_len:
            raise ValueError("超过本模型的教学上下文上限；不自动截断")
        if self.token_embedding.weight.dtype not in (torch.float32, torch.float64):
            raise ValueError("完整教学模型仅支持 FP32/FP64；半精度只用于独立组件实验")
        if torch.is_autocast_enabled(ids.device.type):
            raise ValueError("本课未实现 autocast 路径")

    def forward(self, input_ids: torch.Tensor, *, position_offset: int = 0) -> torch.Tensor:
        self._input_check(input_ids)
        nonnegative_int(position_offset, "position_offset")
        allowed = attention_allowed(input_ids != PAD_ID)
        positions = position_offset + torch.arange(input_ids.shape[1], device=input_ids.device)
        x = self.embedding_dropout(self.token_embedding(input_ids))
        for block in self.blocks:
            x = block(x, positions, allowed)
        return self.lm_head(self.final_norm(x))

    def _cache_check(self, ids, cache, offset):
        self._input_check(ids)
        if any(m.training for m in self.modules()) or torch.is_grad_enabled():
            raise ValueError("缓存路径只接受 eval + torch.no_grad() 推理")
        if bool((ids == PAD_ID).any()):
            raise ValueError("缓存路径不接受 padding；不同长度提示请分开处理")
        if offset is not None:
            nonnegative_int(offset, "position_offset")
        if cache is None:
            return 0, (0 if offset is None else offset)
        if (not isinstance(cache, RotaryCache) or not isinstance(cache.layers, tuple)
                or len(cache.layers) != self.cfg.n_layers):
            raise ValueError("缓存必须是 RotaryCache，且层数与模型一致")
        if cache.owner != id(self) or cache.rope_base != self.cfg.rope_base:
            raise ValueError("缓存所属模型或 RoPE 配置不一致，请重新 prefill")
        nonnegative_int(cache.position_offset, "cache.position_offset")
        if offset is not None and offset != cache.position_offset:
            raise ValueError("不能中途改变缓存的起始位置；请重新 prefill")
        lengths = []
        for pair in cache.layers:
            if not isinstance(pair, tuple) or len(pair) != 2:
                raise ValueError("每层缓存必须是 (K,V)")
            k, v = pair
            if (not isinstance(k, torch.Tensor) or not isinstance(v, torch.Tensor)
                    or k.ndim != 4 or k.shape != v.shape or k.shape[0] != ids.shape[0]
                    or k.shape[1] != self.cfg.n_heads or k.shape[-1] != self.cfg.d_model // self.cfg.n_heads
                    or k.shape[-2] <= 0):
                raise ValueError("缓存需要同形状 [B,H,P,d_head]，P>0")
            for t in pair:
                if (t.device != ids.device or t.dtype != self.token_embedding.weight.dtype
                        or t.requires_grad):
                    raise ValueError("缓存 device/dtype/求导状态不匹配")
            lengths.append(k.shape[-2])
        if len(set(lengths)) != 1:
            raise ValueError("各层历史长度必须一致")
        return lengths[0], cache.position_offset

    def forward_cached(self, input_ids: torch.Tensor, cache: RotaryCache | None = None, *,
                       position_offset: int | None = None) -> tuple[torch.Tensor, RotaryCache]:
        past_len, offset = self._cache_check(input_ids, cache, position_offset)
        new_len = input_ids.shape[1]
        if past_len + new_len > self.cfg.max_len:
            raise ValueError("历史加新增超过上下文；本课不滑动窗口")
        positions = offset + past_len + torch.arange(new_len, device=input_ids.device)
        x = self.embedding_dropout(self.token_embedding(input_ids))
        updated = []
        for index, block in enumerate(self.blocks):
            past = None if cache is None else cache.layers[index]
            delta, kv = block.attn.forward_cached(block.norm1(x), positions, past)
            x = x + block.dropout(delta)
            x = x + block.dropout(block.ffn(block.norm2(x)))
            updated.append(kv)
        new_cache = RotaryCache(tuple(updated), offset, self.cfg.rope_base, id(self))
        return self.lm_head(self.final_norm(x)), new_cache

### 5.1 同一模型的完整与分块计算
替换逐位置 norm/FFN 后，也要重新测试整个模型，不能因为注意力代码没变就跳过因果性和缓存检查。

模型配置中记录 `norm_kind`、`ffn_kind`、`norm_eps`、两种中间宽度。本课明确拒绝旧检查点，不用 `strict=False` 把漏加载的权重伪装成成功。已有模型改成新结构后，需要明确的重新训练或适配方法；这里从随机参数训练。

In [15]:
probe_cfg = ModelConfig(12,d_model=16,n_heads=2,ff_dim=48,swiglu_dim=32)
probe_model = ModernGPT(probe_cfg).eval()
probe_ids = torch.tensor([[BOS_ID,4,5,6]])
with torch.no_grad():
    full = probe_model(probe_ids)
    first, cache = probe_model.forward_cached(probe_ids[:,:2])
    second, cache = probe_model.forward_cached(probe_ids[:,2:],cache)
    incremental = torch.cat((first,second),dim=1)
torch.testing.assert_close(full,incremental,rtol=3e-4,atol=3e-5)
print("输出形状：",tuple(full.shape))
print("最大缓存 logits 差：",(full-incremental).abs().max().item())

输出形状： (1, 4, 12)
最大缓存 logits 差： 2.9802322387695312e-08


## 6. 数据、优化器和指标：四组使用同一份流程
仍使用 96 条原创模板句，先按完整文本划分为 77 条训练、19 条验证，词表只由训练集建立；没有独立测试集。Dataset 只移位一次，标签 padding 为 `-100`。

所有变体：D=48、2层、4头、上下文64、batch16、AdamW 学习率0.003、权重衰减0.01、30轮。Dropout=0 是本课的控制变量设计，不是普遍的训练建议。验证损失按有效 token 总数汇总。

验证集用于选择最佳轮次，不能把最佳验证成绩当成独立测试成绩。所有样本来自同一模板分布，这不是通用语言能力评测。

In [16]:
def demo_lines() -> list[str]:
    """96 条原创模板句；目的只是验证训练流程，不是中文能力基准。"""
    subjects = ["我", "你", "小明", "小红", "同学", "老师", "我们", "他们"]
    times = ["今天", "明天"]
    activities = ["学习数学", "编写程序", "阅读故事", "整理笔记", "检查数据", "训练模型"]
    return [s + t + "正在" + a + "。" for s in subjects for t in times for a in activities]

def clean_texts(lines: list[str]) -> list[str]:
    if not isinstance(lines, list) or any(not isinstance(t, str) for t in lines):
        raise ValueError("语料必须是字符串列表")
    texts, seen = [], set()
    for line in lines:
        text = line.strip()
        if text and text not in seen:
            texts.append(text)
            seen.add(text)
    return texts

def split_texts(texts: list[str], seed: int = 17) -> tuple[list[str], list[str]]:
    if (not isinstance(texts, list) or len(texts) < 5
            or any(not isinstance(t, str) or not t for t in texts)):
        raise ValueError("清理后至少需要 5 条不同的非空文本")
    if len(set(texts)) != len(texts):
        raise ValueError("请先按完整文本去重，再切分")
    shuffled = list(texts)
    random.Random(seed).shuffle(shuffled)
    n_val = max(1, len(texts) // 5)
    return shuffled[n_val:], shuffled[:n_val]

def build_vocab(train_texts: list[str]) -> tuple[dict[str, int], list[str]]:
    if not train_texts or any(not isinstance(t, str) for t in train_texts):
        raise ValueError("训练文本必须是非空字符串列表")
    # 排序固定编号顺序；只能使用训练侧字符。
    chars = sorted(set("".join(train_texts)))
    id_to_token = SPECIAL_TOKENS + chars
    return {token: i for i, token in enumerate(id_to_token)}, id_to_token

def validate_vocab(id_to_token: list[str]) -> None:
    if (not isinstance(id_to_token, list) or len(id_to_token) <= 4
            or id_to_token[:4] != SPECIAL_TOKENS
            or any(not isinstance(t, str) for t in id_to_token)
            or any(len(t) != 1 for t in id_to_token[4:])
            or len(set(id_to_token)) != len(id_to_token)):
        raise ValueError("词表必须按固定顺序包含四个特殊 token，之后是互不重复的单个字符")

def encode(text: str, token_to_id: dict[str, int]) -> list[int]:
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")
    return [token_to_id.get(char, UNK_ID) for char in text]

class CharDataset(Dataset):
    def __init__(self, texts: list[str], token_to_id: dict[str, int], max_len: int) -> None:
        positive_int(max_len, "max_len")
        if not isinstance(texts, list) or not texts:
            raise ValueError("数据集需要非空文本列表")
        self.sequences = []
        for index, text in enumerate(texts):
            ids = [BOS_ID] + encode(text, token_to_id) + [EOS_ID]
            # 输入和目标比完整序列各少一位；不允许静默截断。
            if len(ids) - 1 > max_len:
                raise ValueError(f"第 {index + 1} 条文本需 {len(ids)-1} 个输入位置，超过 max_len={max_len}；请手动分段或调大上下文")
            self.sequences.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        ids = self.sequences[index]
        return ids[:-1], ids[1:]

def collate_batch(samples: list[tuple[torch.Tensor, torch.Tensor]]) -> dict[str, torch.Tensor]:
    if not samples:
        raise ValueError("不能整理空批次")
    xs, ys = [], []
    for x, y in samples:
        if (x.ndim != 1 or x.numel() == 0 or x.shape != y.shape
                or x.dtype != torch.long or y.dtype != torch.long):
            raise ValueError("每条样本的输入、标签必须是等长非空 long 向量")
        xs.append(x)
        ys.append(y)
    return {
        "input_ids": pad_sequence(xs, batch_first=True, padding_value=PAD_ID),
        "targets": pad_sequence(ys, batch_first=True, padding_value=IGNORE_INDEX),
    }

def make_loader(dataset: Dataset, batch_size: int, shuffle: bool = False, seed: int = 17) -> DataLoader:
    positive_int(batch_size, "batch_size")
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False,
                      num_workers=0, collate_fn=collate_batch,
                      generator=torch.Generator().manual_seed(seed))

def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

def loss_parts(logits: torch.Tensor, targets: torch.Tensor) -> tuple[torch.Tensor, int]:
    if (logits.ndim != 3 or targets.shape != logits.shape[:2]
            or targets.dtype != torch.long or logits.device != targets.device):
        raise ValueError("需要同设备 logits=[B,T,V]、long targets=[B,T]")
    if not bool(torch.isfinite(logits).all()):
        raise ValueError("logits 包含非有限值")
    valid = targets != IGNORE_INDEX
    count = int(valid.sum().item())
    if count == 0:
        raise ValueError("本批没有有效目标，不能计算平均损失")
    if bool(((targets[valid] < 0) | (targets[valid] >= logits.shape[-1])).any()):
        raise ValueError("有效目标编号越界")
    total = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1),
                            ignore_index=IGNORE_INDEX, reduction="sum")
    return total, count

def finish_metrics(total_loss: float, total_correct: int, total_tokens: int) -> dict:
    if total_tokens <= 0:
        raise ValueError("没有可评估的有效 token")
    mean = total_loss / total_tokens
    if not math.isfinite(mean):
        raise ValueError("平均损失不是有限数")
    # JSON 不能保存 Inf；异常大损失时不报告 PPL 数值。
    ppl = math.exp(mean) if mean < 700 else None
    return {"loss": mean, "perplexity": ppl, "accuracy": total_correct / total_tokens,
            "tokens": total_tokens}

def evaluate(model: ModernGPT, loader: DataLoader, device: torch.device) -> dict:
    was_training = model.training
    total_loss, total_correct, total_tokens = 0.0, 0, 0
    try:
        model.eval()
        with torch.no_grad():
            for batch in loader:
                inputs, targets = batch["input_ids"].to(device), batch["targets"].to(device)
                logits = model(inputs)
                loss_sum, count = loss_parts(logits, targets)
                valid = targets != IGNORE_INDEX
                total_loss += loss_sum.item()
                total_correct += int(((logits.argmax(-1) == targets) & valid).sum().item())
                total_tokens += count
    finally:
        model.train(was_training)
    return finish_metrics(total_loss, total_correct, total_tokens)

def train_epoch(model: ModernGPT, loader: DataLoader, optimizer: torch.optim.Optimizer,
                device: torch.device, max_grad_norm: float = 1.0) -> dict:
    model.train()
    total_loss, total_correct, total_tokens, steps = 0.0, 0, 0, 0
    for batch in loader:
        inputs, targets = batch["input_ids"].to(device), batch["targets"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss_sum, count = loss_parts(logits, targets)
        (loss_sum / count).backward()
        # 裁剪发生在 backward 之后、step 之前；不是直接截断参数。
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm, error_if_nonfinite=True)
        optimizer.step()
        valid = targets != IGNORE_INDEX
        total_loss += loss_sum.detach().item()
        total_correct += int(((logits.detach().argmax(-1) == targets) & valid).sum().item())
        total_tokens += count
        steps += 1
    result = finish_metrics(total_loss, total_correct, total_tokens)
    result["steps"] = steps
    return result  # 在线指标来自不断改变的参数；选模型用下面的固定参数评估。

def snapshot_state(model: nn.Module) -> dict[str, torch.Tensor]:
    # clone 防止“最佳参数”引用继续被后续训练修改。
    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}

def model_fingerprint(model: nn.Module) -> str:
    digest = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode('utf-8'))
        digest.update(str((tuple(value.shape), value.dtype)).encode('utf-8'))
        digest.update(bytes(value.reshape(-1).view(torch.uint8).tolist()))
    return digest.hexdigest()

In [17]:
def generate_greedy(model: ModernGPT, vocab: list[str], prompt: str = '', *,
                    use_cache: bool = True, max_new_tokens: int = 24) -> dict:
    validate_vocab(vocab)
    if len(vocab) != model.cfg.vocab_size or not isinstance(prompt, str):
        raise ValueError('词表和模型不一致，或 prompt 不是字符串')
    if type(use_cache) is not bool or type(max_new_tokens) is not int or max_new_tokens < 0:
        raise ValueError('use_cache 必须是布尔值；max_new_tokens 必须是非负整数')
    ids = [BOS_ID] + encode(prompt, {t: i for i, t in enumerate(vocab)})
    if len(ids) > model.cfg.max_len:
        raise ValueError('提示超过上下文；不能静默截断')
    device = next(model.parameters()).device
    current = torch.tensor([ids], device=device, dtype=torch.long)
    cache, trace, new_ids = None, [], []
    reason = 'max_new_tokens' if max_new_tokens == 0 else 'context_limit'
    was_training = model.training
    try:
        model.eval()
        with torch.no_grad():
            for step in range(max_new_tokens):
                # 与前课一致：本接口将返回的总长度限制在 max_len，不多吐出第 max_len+1 个编号。
                if current.shape[1] >= model.cfg.max_len:
                    reason = 'context_limit'
                    break
                fed = current if not use_cache or cache is None else current[:, -1:]
                if use_cache:
                    logits, cache = model.forward_cached(fed, cache)
                else:
                    logits = model(fed)
                last = logits[:, -1, :].clone()
                last[:, [BOS_ID, PAD_ID]] = float('-inf')
                next_id = int(last.argmax(-1).item())
                new_ids.append(next_id)
                trace.append({'prediction_step': step, 'input_tokens_this_call': fed.shape[1],
                              'processed_prefix_tokens': current.shape[1],
                              'cache_tokens_after_call': 0 if cache is None else cache.length,
                              'selected_token_id': next_id})
                current = torch.cat((current, current.new_tensor([[next_id]])), dim=1)
                if next_id == EOS_ID:
                    reason = 'eos'
                    break
                reason = 'max_new_tokens'
    finally:
        model.train(was_training)
    visible = ''.join(vocab[i] for i in new_ids if i not in (BOS_ID, EOS_ID, PAD_ID))
    return {'prompt': prompt, 'text': prompt + visible, 'use_cache': use_cache,
            'new_ids': new_ids, 'stop_reason': reason, 'trace': trace,
            'cache_tokens_at_return': 0 if cache is None else cache.length,
            'cache_payload_bytes': cache_payload_bytes(cache),
            'note': '最后选出的 token 尚未送回模型，因此通常未进入 cache；cache 不作为可续接状态返回。'}

def compare_chunked(model: ModernGPT, ids: torch.Tensor, chunks: list[int], *, offset: int = 0) -> dict:
    if not chunks or any(type(c) is not int or c <= 0 for c in chunks) or sum(chunks) != ids.shape[1]:
        raise ValueError("chunks 必须是正整数列表，总和等于 T")
    was_training = model.training
    try:
        model.eval()
        with torch.no_grad():
            full = model(ids, position_offset=offset)
            prefill, full_cache = model.forward_cached(ids, position_offset=offset)
            cache, outputs, start, trace = None, [], 0, []
            for size in chunks:
                logits, cache = model.forward_cached(ids[:,start:start+size], cache, position_offset=offset)
                outputs.append(logits)
                trace.append({"past":start, "new":size, "positions":list(range(offset+start,offset+start+size)),
                              "cache_length":cache.length, "K_shape":list(cache.layers[0][0].shape)})
                start += size
            chunked = torch.cat(outputs, dim=1)
            torch.testing.assert_close(prefill, full, rtol=3e-4, atol=3e-5)
            torch.testing.assert_close(chunked, full, rtol=3e-4, atol=3e-5)
            max_kv = 0.0
            for pair, expected_pair in zip(cache.layers, full_cache.layers):
                for value, expected in zip(pair, expected_pair):
                    torch.testing.assert_close(value, expected, rtol=3e-4, atol=3e-5)
                    max_kv = max(max_kv, (value-expected).abs().max().item())
            shifted = model(ids, position_offset=offset+31)
            torch.testing.assert_close(shifted, full, rtol=3e-4, atol=3e-5)
            return {"chunks":chunks, "position_offset":offset, "trace":trace,
                    "max_logits_error":(chunked-full).abs().max().item(),
                    "max_cache_error":max_kv, "global_shift_logits_error":(shifted-full).abs().max().item(),
                    "cache_bytes":cache_payload_bytes(cache), "rtol":3e-4, "atol":3e-5}
    finally:
        model.train(was_training)

### 6.1 四组对照与共享初始化

| 组名 | 归一化 | FFN |
|---|---|---|
| `layer_gelu` | LayerNorm | GELU |
| `rms_gelu` | RMSNorm | GELU |
| `layer_swiglu` | LayerNorm | SwiGLU |
| `rms_swiglu` | RMSNorm | SwiGLU |

四组的 token embedding、注意力和输出头等共有参数逐项复制，数值相同但不是同一个参数对象。同 FFN 的归一化对照还复制整个 FFN；不同 FFN 的矩阵形状不同，不能声称初始函数相同。

每组使用同种子新建 DataLoader，所以每一轮遍历顺序一致。控制这些因素可以减少混淆，但**单种子、小样本、同一组未分别调优的超参数**仍不足以得出一般优劣结论。

In [18]:
VARIANTS = {'layer_gelu': ('layer', 'gelu'), 'rms_gelu': ('rms', 'gelu'), 'layer_swiglu': ('layer', 'swiglu'), 'rms_swiglu': ('rms', 'swiglu')}

def make_variants(vocab_size: int, seed: int = 21, **config_overrides) -> dict[str, ModernGPT]:
    """在 CPU 初始化，再由调用者移到设备；不同模型不共享参数对象。"""
    models = {}
    for index, (name, (norm, ffn)) in enumerate(VARIANTS.items()):
        with torch.random.fork_rng(devices=[]):
            torch.random.default_generator.manual_seed(seed + index)
            cfg = ModelConfig(vocab_size, norm_kind=norm, ffn_kind=ffn, **config_overrides)
            model = ModernGPT(cfg)
        if models:
            # 相同 FFN 的归一化对照：复制整个共有部分。
            # 不同 FFN 的对照：只复制非 FFN 部分，不声称 FFN 初始函数一致。
            source = models["layer_swiglu"] if name == "rms_swiglu" else models["layer_gelu"]
            source_state = source.state_dict()
            with torch.no_grad():
                for key, tensor in model.state_dict().items():
                    same_ffn = source.cfg.ffn_kind == ffn
                    if ".ffn." in key and not same_ffn:
                        continue
                    if key in source_state and tensor.shape == source_state[key].shape:
                        tensor.copy_(source_state[key])
        models[name] = model
    return models

def initialization_report(models: dict[str, ModernGPT]) -> dict:
    baseline = models["layer_gelu"].state_dict()
    comparisons = {}
    for name, model in models.items():
        common = [key for key, value in model.state_dict().items()
                  if ".ffn." not in key and key in baseline and value.shape == baseline[key].shape]
        matched = all(torch.equal(model.state_dict()[key], baseline[key]) for key in common)
        comparisons[name] = {"common_tensor_count": len(common), "all_equal": matched,
                             "parameters": count_parameters(model),
                             "ffn_weights_per_layer": count_parameters(model.blocks[0].ffn)}
        if not matched:
            raise AssertionError("公共初始参数没有对齐")
    return comparisons

In [19]:
train_preview,val_preview = split_texts(demo_lines(),17)
_,vocab_preview = build_vocab(train_preview)
initial_models = make_variants(len(vocab_preview),SEED)
print("训练/验证/词表大小：",len(train_preview),len(val_preview),len(vocab_preview))
print(json.dumps(initialization_report(initial_models),ensure_ascii=False,indent=2))
del initial_models

训练/验证/词表大小： 77 19 41


{
  "layer_gelu": {
    "common_tensor_count": 28,
    "all_equal": true,
    "parameters": 60096,
    "ffn_weights_per_layer": 18432
  },
  "rms_gelu": {
    "common_tensor_count": 23,
    "all_equal": true,
    "parameters": 59856,
    "ffn_weights_per_layer": 18432
  },
  "layer_swiglu": {
    "common_tensor_count": 28,
    "all_equal": true,
    "parameters": 60096,
    "ffn_weights_per_layer": 18432
  },
  "rms_swiglu": {
    "common_tensor_count": 23,
    "all_equal": true,
    "parameters": 59856,
    "ffn_weights_per_layer": 18432
  }
}


## 7. 运行完整实验
下面的支持函数包含训练、检查点、缓存核对和报告保存。每组最佳权重来自自己的验证最低轮次。检查点只用于模型恢复和生成，不含优化器及完整随机状态，不能精确恢复训练进度。

默认会训练四组。只想先走通新组件时，将顶部 `SINGLE=True`；已有本课检查点时，设置 `CHECKPOINT` 跳过训练。两种方式都不改写输入检查点。

In [20]:
def train_variant(model: ModernGPT, train_texts: list[str], val_texts: list[str],
                  vocab: list[str], device: torch.device, *, epochs: int, seed: int,
                  name: str) -> dict:
    positive_int(epochs, "epochs")
    mapping = {t: i for i, t in enumerate(vocab)}
    train_data = CharDataset(train_texts, mapping, model.cfg.max_len)
    val_data = CharDataset(val_texts, mapping, model.cfg.max_len)
    # 每个变体都使用新的同种子 loader，所以各轮样本顺序一致。
    train_loader = make_loader(train_data, 16, True, seed)
    train_eval = make_loader(train_data, 16)
    val_loader = make_loader(val_data, 16)
    set_seed(seed)
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.01)
    before = {"train": evaluate(model, train_eval, device),
              "validation": evaluate(model, val_loader, device)}
    best_state = snapshot_state(model)
    best_epoch, best_loss = 0, before["validation"]["loss"]
    history = []
    for epoch in range(1, epochs + 1):
        online = train_epoch(model, train_loader, optimizer, device)
        row = {"epoch": epoch, "online_train": online,
               "train": evaluate(model, train_eval, device),
               "validation": evaluate(model, val_loader, device)}
        history.append(row)
        if row["validation"]["loss"] < best_loss:
            best_loss, best_epoch = row["validation"]["loss"], epoch
            best_state = snapshot_state(model)
        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(f"{name:13s} epoch={epoch:02d} train={row['train']['loss']:.4f} val={row['validation']['loss']:.4f}")
    model.load_state_dict(best_state)
    model.eval()
    model.zero_grad(set_to_none=True)
    return {"seed": seed, "epochs": epochs, "config": asdict(model.cfg),
            "parameters": count_parameters(model), "before": before, "history": history,
            "best_epoch": best_epoch,
            "selected_train": evaluate(model, train_eval, device),
            "selected_validation": evaluate(model, val_loader, device),
            "updates": sum(row["online_train"]["steps"] for row in history)}

def save_checkpoint(path: Path, model: ModernGPT, vocab: list[str], best_epoch: int) -> None:
    validate_vocab(vocab)
    if len(vocab) != model.cfg.vocab_size:
        raise ValueError("词表大小不匹配")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"family": CHECKPOINT_FAMILY, "config": asdict(model.cfg), "vocab": vocab,
                "state_dict": snapshot_state(model), "best_epoch": best_epoch,
                "purpose": "inference_only_not_exact_training_resume"}, path)

def load_checkpoint(path: Path, device: torch.device) -> tuple[ModernGPT, list[str], dict]:
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict) or payload.get("family") != CHECKPOINT_FAMILY:
        raise ValueError("需要第21课专属检查点；旧架构不能靠 strict=False 自动转换")
    vocab, cfg = payload["vocab"], ModelConfig(**payload["config"])
    validate_vocab(vocab)
    if cfg.vocab_size != len(vocab):
        raise ValueError("词表与模型配置不一致")
    with torch.random.fork_rng(devices=[]):
        model = ModernGPT(cfg)
    model.load_state_dict(payload["state_dict"], strict=True)
    model.to(device).eval()
    return model, vocab, {"source": "lesson21_checkpoint", "best_epoch": payload["best_epoch"]}

def model_probe(model: ModernGPT, vocab: list[str]) -> dict:
    device = next(model.parameters()).device
    # 支持本课检查点，不假定任意词表里都有某句中文。
    second_id = 5 if model.cfg.vocab_size > 5 else 4
    ids = torch.tensor([[BOS_ID, 4, second_id, 4, second_id]], dtype=torch.long, device=device)
    before_hash = model_fingerprint(model)
    chunked = compare_chunked(model, ids, [2, 1, 2], offset=7)
    was_training = model.training
    try:
        model.eval()
        with torch.no_grad():
            original = model(ids)
            changed = ids.clone()
            changed[:, -1] = 6 if model.cfg.vocab_size > 6 else 4
            future_error = (original[:, :-1] - model(changed)[:, :-1]).abs().max().item()
            torch.testing.assert_close(original[:, :-1], model(changed)[:, :-1])
    finally:
        model.train(was_training)
    samples = []
    for prompt in ("我今天正在", "小明明天正在"):
        cached = generate_greedy(model, vocab, prompt, use_cache=True)
        plain = generate_greedy(model, vocab, prompt, use_cache=False)
        equal = cached["new_ids"] == plain["new_ids"] and cached["stop_reason"] == plain["stop_reason"]
        # 近似并列 logits 可能造成分叉；等价性的核心保证是带容差的 logits 检查。
        samples.append({"prompt": prompt, "cache": cached, "full": plain, "same_token_ids": equal})
    unchanged = model_fingerprint(model) == before_hash
    if not unchanged:
        raise AssertionError("推理修改了参数")
    return {"chunked": chunked, "future_logits_error": future_error,
            "samples": samples, "weights_unchanged": unchanged}

def run_experiment(device: torch.device, output_dir: Path, *, epochs: int = 30,
                   seed: int = 21, single: bool = False,
                   checkpoint: Path | None = None) -> dict:
    output_dir = Path(output_dir)
    if checkpoint is not None:
        # 不把输入检查点当输出覆盖。loaded 只写 JSON 报告，不重存输入文件。
        model, vocab, source = load_checkpoint(Path(checkpoint), device)
        models = {"loaded": model}
        training = {"loaded": source}
        initial = None
        split = None
    else:
        train_texts, val_texts = split_texts(demo_lines(), seed=17)
        _, vocab = build_vocab(train_texts)
        models = make_variants(len(vocab), seed)
        initial = initialization_report(models)
        if single:
            models = {"rms_swiglu": models["rms_swiglu"]}
        training = {}
        for name, model in models.items():
            training[name] = train_variant(model, train_texts, val_texts, vocab, device,
                                           epochs=epochs, seed=seed, name=name)
        split = {"split_seed": 17, "train_texts": train_texts, "validation_texts": val_texts,
                 "vocab": vocab, "validation_used_for_selection": True, "has_test_set": False}
    output_dir.mkdir(parents=True, exist_ok=True)
    probes = {}
    for name, model in models.items():
        probes[name] = model_probe(model, vocab)
        if checkpoint is None:
            path = output_dir / (name + ".pt")
            save_checkpoint(path, model, vocab, training[name]["best_epoch"])
            restored, restored_vocab, _ = load_checkpoint(path, device)
            ids = torch.tensor([[BOS_ID, 4, 5]], device=device)
            with torch.no_grad():
                torch.testing.assert_close(model(ids), restored(ids), rtol=0, atol=0)
            probes[name]["reload_equal"] = restored_vocab == vocab
    summary = {"environment": {"python": platform.python_version(), "torch": str(torch.__version__),
                               "device": str(device), "cuda_available": torch.cuda.is_available(),
                               "threads": torch.get_num_threads()},
               "initialization": initial, "split": split, "components": component_report(),
               "probes": probes,
               "limitations": ["单种子、人工模板，验证集用于选轮次，没有独立测试集",
                               "相同 FFN 权重数量不代表相同延迟、峰值显存或整个模型参数量",
                               "学习公式和实现，不据此宣称 RMSNorm/SwiGLU 一定更好或更快",
                               "完整模型 FP32；组件半精度实验不代表支持全模型 AMP",
                               "不是 Llama 检查点兼容实现；未包含 GQA、量化或服务系统"]}
    for name, content in (("metrics.json", summary), ("training.json", training)):
        (output_dir / name).write_text(json.dumps(content, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    for name, record in training.items():
        if "selected_validation" in record:
            print(f"{name}: 参数={record['parameters']}, 最佳轮次={record['best_epoch']}, "
                  f"验证损失={record['selected_validation']['loss']:.6f}")
    print("输出目录：", output_dir.resolve())
    return {"summary": summary, "training": training}

In [21]:
experiment = run_experiment(
    device, OUTPUT_DIR, epochs=EPOCHS, seed=SEED, single=SINGLE,
    checkpoint=None if CHECKPOINT is None else Path(CHECKPOINT),
)

layer_gelu    epoch=01 train=2.8275 val=2.8302


layer_gelu    epoch=10 train=0.4637 val=0.4945


layer_gelu    epoch=20 train=0.4029 val=0.4281


layer_gelu    epoch=30 train=0.3961 val=0.4518
rms_gelu      epoch=01 train=2.8191 val=2.8220


rms_gelu      epoch=10 train=0.4639 val=0.4924


rms_gelu      epoch=20 train=0.4017 val=0.4339


... (4 outputs left out)

rms_swiglu    epoch=10 train=0.4525 val=0.4846


rms_swiglu    epoch=20 train=0.4034 val=0.4310


rms_swiglu    epoch=30 train=0.3946 val=0.4371


layer_gelu: 参数=60096, 最佳轮次=20, 验证损失=0.428052
rms_gelu: 参数=59856, 最佳轮次=20, 验证损失=0.433949
layer_swiglu: 参数=60096, 最佳轮次=26, 验证损失=0.430763
rms_swiglu: 参数=59856, 最佳轮次=26, 验证损失=0.430409
输出目录： /mnt/data/_work21/nb_run/lesson_21_outputs


### 7.1 读取结果，不预先宣布哪个模型获胜
下面展示实际运行的数据，而不是写死的“预期最优”数字。比较固定轮次与各自选中轮次会回答不同问题；完整历史已经保存在 `training.json`。

在本次交付前的 CPU 单种子运行中，四组所选验证损失约为 0.428–0.434，彼此很接近，不能宣称 RMSNorm+SwiGLU 必然胜出。不同系统、PyTorch 版本、种子和硬件可能改变具体结果与排名。

In [22]:
for name,row in experiment["training"].items():
    if "selected_validation" in row:
        print(f"{name:13s} 参数={row['parameters']:6d}  最佳轮次={row['best_epoch']:2d}  "
              f"验证CE={row['selected_validation']['loss']:.6f}  最后一轮CE={row['history'][-1]['validation']['loss']:.6f}")
for name,probe in experiment["summary"]["probes"].items():
    print(name,"缓存最大差=",probe['chunked']['max_logits_error'],
          "权重未修改=",probe['weights_unchanged'])
    for sample in probe['samples']:
        print(" ",sample['cache']['text'],"完整/缓存编号一致=",sample['same_token_ids'])

layer_gelu    参数= 60096  最佳轮次=20  验证CE=0.428052  最后一轮CE=0.451772
rms_gelu      参数= 59856  最佳轮次=20  验证CE=0.433949  最后一轮CE=0.451554
layer_swiglu  参数= 60096  最佳轮次=26  验证CE=0.430763  最后一轮CE=0.440325
rms_swiglu    参数= 59856  最佳轮次=26  验证CE=0.430409  最后一轮CE=0.437144
layer_gelu 缓存最大差= 1.430511474609375e-06 权重未修改= True
  我今天正在整理笔记。 完整/缓存编号一致= True
  小明明天正在整理笔记。 完整/缓存编号一致= True
rms_gelu 缓存最大差= 1.430511474609375e-06 权重未修改= True
  我今天正在整理笔记。 完整/缓存编号一致= True
  小明明天正在整理笔记。 完整/缓存编号一致= True
layer_swiglu 缓存最大差= 1.9073486328125e-06 权重未修改= True
  我今天正在学习数学。 完整/缓存编号一致= True
  小明明天正在检查数据。 完整/缓存编号一致= True
rms_swiglu 缓存最大差= 1.1920928955078125e-06 权重未修改= True
  我今天正在学习数学。 完整/缓存编号一致= True
  小明明天正在检查数据。 完整/缓存编号一致= True


### 7.2 保存文件和边界
默认生成 `layer_gelu.pt`、`rms_gelu.pt`、`layer_swiglu.pt`、`rms_swiglu.pt`，以及 `metrics.json` 和 `training.json`。单模型模式只生成对应检查点；加载模式只输出 JSON。

缓存对照核对了原始 logits、各层 K/V 以及固定提示的贪心结果。因为浮点计算次序不同，使用误差容限；近似并列 logits 可能使后续生成分叉，不能把一段生成文本一致当成全部证明。

全模型未实现 AMP、量化、GQA、滑动窗口或生产服务。本节也没有性能基准；手写 RMSNorm 的运算少一些，不保证快于库中融合优化过的 LayerNorm。

## 8. 自动检查
以下测试覆盖公式、归一化轴、官方梯度、低精度平方、门控分支、参数预算、公共初始化、因果性、padding、缓存、残差和重载。每项检查失败立即停止，不使用可能被 `python -O` 移除的 `assert`。

先尝试预测：若误把 RMSNorm 改成“先减均值”，哪一个最简单的常量测试会失败？若只对 gate 做 sigmoid，而非 SiLU，哪个公式测试能发现？

In [23]:
class Checks:
    def __init__(self):
        self.names = []

    def close(self, name, actual, expected, **kwargs):
        torch.testing.assert_close(actual, expected, **kwargs)
        self.names.append(name)

    def true(self, name, condition):
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)

    def raises(self, name, fn, expected=ValueError):
        try:
            fn()
        except expected:
            self.names.append(name)
        else:
            raise AssertionError(f"{name}：没有抛出 {expected.__name__}")

def run_checks(device: torch.device = torch.device("cpu")) -> list[str]:
    if not hasattr(F, "rms_norm"):
        raise RuntimeError("官方核对需要支持 F.rms_norm 的 PyTorch；课程在 PyTorch 2.10.0 CPU 验证")
    c = Checks()
    set_seed(2101)
    dtype = torch.float64
    x = torch.tensor([3., 4.], dtype=dtype, device=device)
    w = torch.ones_like(x)
    eps = 1e-6
    c.close("RMSNorm 手算 sqrt(12.5+eps)", rms_norm(x,w,eps), x/math.sqrt(12.5+eps))
    c.true("RMSNorm 不强制均值为0", rms_norm(x,w,eps).mean() > 0.9)
    c.close("RMSNorm 零输入", rms_norm(torch.zeros_like(x),w,eps), torch.zeros_like(x))
    constant = torch.full((4,),3.,dtype=dtype,device=device)
    weight4 = torch.ones_like(constant)
    c.close("LayerNorm 常量为0", F.layer_norm(constant,(4,),eps=eps), torch.zeros_like(constant))
    c.close("RMSNorm 常量接近1", rms_norm(constant,weight4,eps), constant/math.sqrt(9+eps))
    y = rms_norm(x,w,eps)
    c.close("eps 下均方不精确等于1", y.square().mean(), x.square().mean()/(x.square().mean()+eps))
    z = torch.tensor([1.,2.,3.,4.],dtype=dtype,device=device)
    c.close("LN 平移不变", F.layer_norm(z,(4,),eps=eps),F.layer_norm(z+10,(4,),eps=eps))
    c.true("RMSNorm 通常不平移不变", not torch.allclose(rms_norm(z,weight4),rms_norm(z+10,weight4)))
    c.close("大信号正缩放近似不变", rms_norm(z,weight4,1e-12),rms_norm(z*7,weight4,1e-12))
    tiny=z*1e-5
    c.true("eps 破坏精确缩放不变", not torch.allclose(rms_norm(tiny,weight4),rms_norm(7*tiny,weight4)))
    c.close("负缩放改变符号", rms_norm(-z,weight4), -rms_norm(z,weight4))
    batch = torch.randn(2,3,4,device=device,dtype=dtype,requires_grad=True)
    gamma = torch.tensor([0.5,1.,1.5,2.],device=device,dtype=dtype,requires_grad=True)
    custom = rms_norm(batch,gamma,eps)
    official=F.rms_norm(batch,(4,),gamma,eps)
    c.close("官方 RMSNorm 前向",custom,official,rtol=1e-11,atol=1e-11)
    upstream=torch.randn_like(batch)
    actual_grads=torch.autograd.grad((custom*upstream).sum(),(batch,gamma),retain_graph=True)
    expected_grads=torch.autograd.grad((official*upstream).sum(),(batch,gamma))
    c.close("RMSNorm 输入梯度",actual_grads[0],expected_grads[0],rtol=1e-10,atol=1e-10)
    c.close("RMSNorm gamma梯度",actual_grads[1],expected_grads[1],rtol=1e-10,atol=1e-10)
    nc=torch.randn(2,4,3,dtype=dtype,device=device).transpose(1,2)
    c.close("RMSNorm 非连续输入",rms_norm(nc,gamma),F.rms_norm(nc,(4,),gamma,eps))
    c.close("独立处理token",rms_norm(batch.detach(),gamma)[0,1],rms_norm(batch.detach()[0,1],gamma))
    norm=RMSNorm(4,eps).to(device=device,dtype=dtype)
    c.true("RMSNorm 只有D个参数",count_parameters(norm)==4)
    c.true("RMSNorm 无beta",list(norm.state_dict())==["weight"])
    norm.train(); train_y=norm(nc); norm.eval()
    c.close("RMSNorm train eval一致",norm(nc),train_y)
    c.true("RMSNorm FP64梯度数值校验",torch.autograd.gradcheck(
        lambda a,b:rms_norm(a,b,eps),
        (torch.randn(2,3,device=device,dtype=dtype,requires_grad=True),
         torch.ones(3,device=device,dtype=dtype,requires_grad=True))))
    for low in (torch.float16,torch.bfloat16):
        low_x=torch.tensor([[300.,400.]],device=device,dtype=low)
        got=rms_norm(low_x,torch.ones(2,device=device),eps)
        expected=rms_norm(low_x.float(),torch.ones(2,device=device),eps).to(low)
        c.close(f"{low}升精度后还原",got,expected,rtol=0,atol=0)
        c.true(f"{low}输出有限且dtype保留",bool(torch.isfinite(got).all()) and got.dtype==low)
    bad_half=torch.tensor([300.,400.],dtype=torch.float16,device=device)
    c.true("FP16直接平方可溢出",not bool(torch.isfinite(bad_half.square()).all()))
    c.raises("拒绝整数归一化输入",lambda:rms_norm(torch.ones(2,dtype=torch.long,device=device),w))
    c.raises("拒绝不匹配gamma",lambda:rms_norm(x,weight4))
    c.raises("拒绝空输入",lambda:rms_norm(torch.empty(0,device=device),torch.empty(0,device=device)))
    for invalid in (0,-1,float('nan'),True):
        c.raises(f"拒绝eps={invalid}",lambda invalid=invalid:rms_norm(x,w,invalid))

    values=torch.tensor([-2.,-1.,0.,1.,2.],device=device,dtype=dtype)
    c.close("SiLU定义",F.silu(values),values*torch.sigmoid(values))
    c.true("SiLU门控可负且可大于1",F.silu(values).min()<0 and F.silu(values).max()>1)
    ffn=SwiGLUFFN(4,6).to(device=device,dtype=dtype)
    sample=torch.randn(2,3,4,device=device,dtype=dtype,requires_grad=True)
    actual=ffn(sample)
    gate=sample@ffn.gate_proj.weight.T
    value=sample@ffn.up_proj.weight.T
    expected=(gate*torch.sigmoid(gate)*value)@ffn.down_proj.weight.T
    c.close("SwiGLU逐矩阵公式",actual,expected)
    c.true("SwiGLU保持[B,T,D]",actual.shape==sample.shape)
    params=tuple(ffn.parameters())
    grad1=torch.autograd.grad(actual.square().sum(),(sample,*params),retain_graph=True)
    grad2=torch.autograd.grad(expected.square().sum(),(sample,*params))
    for i,(g1,g2) in enumerate(zip(grad1,grad2)):
        c.close(f"SwiGLU输入或参数梯度{i}",g1,g2,rtol=1e-9,atol=1e-10)
    c.true("SwiGLU三矩阵都有梯度",all(bool(torch.isfinite(g).all()) and bool(g.abs().sum()>0) for g in grad1[1:]))
    c.true("SwiGLU无偏置",all(not name.endswith('bias') for name,_ in ffn.named_parameters()))
    c.true("SwiGLU参数3DH",count_parameters(ffn)==3*4*6)
    c.true("GELU参数2DM",count_parameters(GELUFFN(4,6))==2*4*6)
    changed=sample.detach().clone();changed[0,0]+=100
    c.close("FFN不会跨token传递",ffn(changed)[0,1:],ffn(sample.detach())[0,1:])
    for branch in ('gate_proj','up_proj','down_proj'):
        test=SwiGLUFFN(4,6).to(device=device,dtype=dtype)
        with torch.no_grad():getattr(test,branch).weight.zero_()
        c.close(f"{branch}为0则输出为0",test(sample),torch.zeros_like(sample))
    c.true("D48精确预算",parameter_budget(48,192,128)['gelu_weights']==18432==parameter_budget(48,192,128)['swiglu_weights'])
    c.true("D64对齐8得到176",swiglu_budget_width(64,8)==176)
    c.true("对齐后不再同参数",parameter_budget(64,256,176)['swiglu_weights']==33792)
    c.true("不对齐D48得到128",swiglu_budget_width(48)==128)
    c.raises("拒绝非法multiple",lambda:swiglu_budget_width(48,0))
    c.raises("拒绝布尔dim",lambda:swiglu_budget_width(True))

    models=make_variants(12,seed=21,d_model=16,n_heads=2,n_layers=2,ff_dim=48,swiglu_dim=32,max_len=16)
    init=initialization_report(models)
    c.true("四组公共初始化一致",all(v['all_equal'] for v in init.values()))
    c.true("同FFN跨norm初始权重一致",all(torch.equal(models['layer_swiglu'].state_dict()[k],v)
        for k,v in models['rms_swiglu'].state_dict().items() if '.ffn.' in k))
    c.true("不同模型不共享Parameter对象",models['layer_gelu'].token_embedding.weight.data_ptr()!=models['rms_gelu'].token_embedding.weight.data_ptr())
    c.true("总参数差来自5组beta",count_parameters(models['layer_gelu'])-count_parameters(models['rms_gelu'])==5*16)
    c.true("每层FFN参数精确相等",count_parameters(models['layer_gelu'].blocks[0].ffn)==count_parameters(models['layer_swiglu'].blocks[0].ffn))
    ids=torch.tensor([[BOS_ID,4,5,6],[BOS_ID,7,8,PAD_ID]],device=device)
    labels=torch.tensor([[4,5,6,EOS_ID],[7,8,EOS_ID,IGNORE_INDEX]],device=device)
    for name,model in models.items():
        model.to(device).eval()
        logits=model(ids)
        c.true(f"{name}输出形状",logits.shape==(2,4,12))
        total,n=loss_parts(logits,labels); (total/n).backward()
        c.true(f"{name}全部参数有限梯度",all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in model.parameters()))
        short=ids[:1,:3]
        with torch.no_grad():
            padded=torch.cat((short,torch.full((1,2),PAD_ID,device=device,dtype=torch.long)),dim=1)
            c.close(f"{name}padding不改变有效前缀",model(padded)[:,:3],model(short),rtol=2e-4,atol=2e-5)
            altered=ids[:1].clone(); altered[:,-1]=9
            c.close(f"{name}因果性",model(ids[:1])[:,:-1],model(altered)[:,:-1])
        report=compare_chunked(model,ids[:1],[1,2,1],offset=5)
        c.true(f"{name}缓存与整段一致",report['max_logits_error']<3e-5)
        c.true(f"{name}有限损失",math.isfinite(total.item()) and n==7)
    model=models['rms_swiglu']
    full=model(ids).detach()
    with torch.no_grad():
        _,old_cache=model.forward_cached(ids[:1,:2])
        before=tuple(t.clone() for pair in old_cache.layers for t in pair)
        _,new_cache=model.forward_cached(ids[:1,2:],old_cache)
        c.true("追加后新旧缓存长度",old_cache.length==2 and new_cache.length==4)
        c.true("旧缓存不被改写",all(torch.equal(a,b) for a,b in zip(before,(t for pair in old_cache.layers for t in pair))))
        c.raises("不混用不同模型缓存",lambda:models['layer_gelu'].forward_cached(ids[:1,:1],old_cache))
    c.raises("缓存不构建训练图",lambda:model.forward_cached(ids[:1]))
    model.train()
    with torch.no_grad():
        c.raises("缓存需要eval",lambda:model.forward_cached(ids[:1]))
    model.eval()
    zero_block=ModernBlock(model.cfg).to(device).eval()
    with torch.no_grad():
        zero_block.attn.out_proj.weight.zero_();zero_block.attn.out_proj.bias.zero_()
        zero_block.ffn.down_proj.weight.zero_()
    state=torch.randn(1,3,16,device=device,requires_grad=True)
    result=zero_block(state,torch.arange(3,device=device),offset_causal_mask(0,3,device))
    c.close("零分支保留残差输入",result,state,rtol=0,atol=0)
    result.sum().backward()
    c.close("零分支保留直通梯度",state.grad,torch.ones_like(state))
    with tempfile.TemporaryDirectory() as tmp:
        path=Path(tmp)/'model.pt'
        vocab=SPECIAL_TOKENS+list('abcdefgh')
        save_checkpoint(path,model,vocab,2)
        loaded,got_vocab,meta=load_checkpoint(path,device)
        c.true("检查点词表和配置",got_vocab==vocab and loaded.cfg==model.cfg and meta['best_epoch']==2)
        with torch.no_grad():c.close("重载后logits一致",loaded(ids),model(ids),rtol=0,atol=0)
        bad=Path(tmp)/'old.pt';torch.save({'family':'lesson20_rope_adjacent_v1'},bad)
        c.raises("拒绝旧架构检查点",lambda:load_checkpoint(bad,device))
    c.raises("拒绝非法norm配置",lambda:ModelConfig(12,norm_kind='batch'))
    c.raises("拒绝非法FFN配置",lambda:ModelConfig(12,ffn_kind='sigmoid'))
    print(f"通过固定检查：{len(c.names)} 项")
    return c.names

In [24]:
check_names = run_checks(device)
print("本次检查数量：",len(check_names))

通过固定检查：95 项
本次检查数量： 95


## 9. 独立练习区：先停在这里
每题检查开关默认关闭。完成实现后，把对应 `RUN_EXERCISE_*` 改为 `True`。输入前置条件已说明，先专注数学正确性；需要时再增加与完整脚本一样的类型与形状检查。

### 练习 A：`rms_norm_ex(x, weight, eps)`
输入为同设备浮点张量 `x=[...,D]`、`weight=[D]`、`eps>0`。要求只沿最后轴计算、不减均值、保留输入 dtype、FP16/BF16 先升 FP32、FP64 保留、梯度不断开。不能直接调用 `F.rms_norm` 或本课成品函数。

In [25]:
def check_exercise_a(fn):
    x = torch.tensor([[[1., -2., 3., 4.], [3., 3., 3., 3.]]], dtype=torch.float64, requires_grad=True)
    weight = torch.tensor([0.5, 1., 1.5, 2.], dtype=torch.float64, requires_grad=True)
    expected = F.rms_norm(x, (4,), weight, 1e-6)
    actual = fn(x, weight, 1e-6)
    torch.testing.assert_close(actual, expected, rtol=1e-10, atol=1e-10)
    upstream = torch.tensor([[[1., -1., 2., 0.5], [0.1, 0.2, -0.1, 0.4]]], dtype=torch.float64)
    a = torch.autograd.grad((actual * upstream).sum(), (x, weight), retain_graph=True)
    b = torch.autograd.grad((expected * upstream).sum(), (x, weight))
    for ga, gb in zip(a, b):
        torch.testing.assert_close(ga, gb, rtol=1e-9, atol=1e-10)
    half = torch.tensor([300., 400.], dtype=torch.float16)
    got = fn(half, torch.ones(2), 1e-6)
    ref = F.rms_norm(half.float(), (2,), torch.ones(2), 1e-6).half()
    torch.testing.assert_close(got, ref, rtol=0, atol=0)
    zero = torch.zeros(4)
    torch.testing.assert_close(fn(zero, torch.ones(4), 1e-6), zero)
    print("练习 A：前向、输入/gamma 梯度、零向量与 FP16 溢出案例均通过")

def rms_norm_ex(x, weight, eps=1e-6):
    raise NotImplementedError("请在这里实现 RMSNorm")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    check_exercise_a(rms_norm_ex)
else:
    print("练习A未启用：请完成函数后再打开检查")

练习A未启用：请完成函数后再打开检查


### 练习 B：`swiglu_ex(x, w_gate, w_up, w_down)`
输入 `x=[B,T,D]`；三个权重为 `[H,D]`、`[H,D]`、`[D,H]`，同 dtype/设备，全部无偏置。实现上投影、SiLU、逐元素乘法、下投影。可用 `F.linear` 和 `F.silu`，不能调用完整 `SwiGLUFFN`。输出和四组梯度都应符合参考公式。

想一想：去掉乘法，改成 `gate+value`，为什么形状仍合法，却不是本课的网络？

In [26]:
def check_exercise_b(fn):
    g = torch.Generator().manual_seed(2102)
    x = torch.randn(2, 3, 4, generator=g, dtype=torch.float64, requires_grad=True)
    wg = torch.randn(6, 4, generator=g, dtype=torch.float64, requires_grad=True)
    wu = torch.randn(6, 4, generator=g, dtype=torch.float64, requires_grad=True)
    wd = torch.randn(4, 6, generator=g, dtype=torch.float64, requires_grad=True)
    gate = F.linear(x, wg)
    expected = F.linear(gate * torch.sigmoid(gate) * F.linear(x, wu), wd)
    actual = fn(x, wg, wu, wd)
    torch.testing.assert_close(actual, expected)
    a = torch.autograd.grad(actual.square().mean(), (x, wg, wu, wd), retain_graph=True)
    b = torch.autograd.grad(expected.square().mean(), (x, wg, wu, wd))
    for ga, gb in zip(a, b):
        torch.testing.assert_close(ga, gb)
    for which in range(3):
        weights = [wg.detach().clone(), wu.detach().clone(), wd.detach().clone()]
        weights[which].zero_()
        torch.testing.assert_close(fn(x.detach(), *weights), torch.zeros_like(x))
    print("练习 B：三矩阵公式、所有梯度和三个零分支案例均通过")

def swiglu_ex(x, w_gate, w_up, w_down):
    raise NotImplementedError("请在这里实现 SwiGLU")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    check_exercise_b(swiglu_ex)
else:
    print("练习B未启用：请完成函数后再打开检查")

练习B未启用：请完成函数后再打开检查


### 练习 C：`pre_rms_ffn_ex(...)`
完成：
\[
y=x+\operatorname{SwiGLU}(\operatorname{RMSNorm}(x)).
\]
可使用自己完成的 A、B，或手写所有步骤。要求原始 `x` 作为残差、返回值保留梯度。检查包括正常参数下所有梯度，以及 `w_down=0` 时的恒等路径。

这道题只实现 Block 的 FFN 子层，不包含注意力。

In [27]:
def check_exercise_c(fn):
    g = torch.Generator().manual_seed(2103)
    x = torch.randn(2, 3, 4, generator=g, dtype=torch.float64, requires_grad=True)
    gamma = torch.tensor([0.5, 1., 1.5, 2.], dtype=torch.float64, requires_grad=True)
    wg = torch.randn(6, 4, generator=g, dtype=torch.float64, requires_grad=True)
    wu = torch.randn(6, 4, generator=g, dtype=torch.float64, requires_grad=True)
    wd = torch.randn(4, 6, generator=g, dtype=torch.float64, requires_grad=True)
    n = F.rms_norm(x, (4,), gamma, 1e-6)
    expected = x + F.linear(F.silu(F.linear(n, wg)) * F.linear(n, wu), wd)
    actual = fn(x, gamma, wg, wu, wd, 1e-6)
    torch.testing.assert_close(actual, expected)
    a = torch.autograd.grad(actual.square().mean(), (x, gamma, wg, wu, wd), retain_graph=True)
    b = torch.autograd.grad(expected.square().mean(), (x, gamma, wg, wu, wd))
    for ga, gb in zip(a, b):
        torch.testing.assert_close(ga, gb)
    zero_down = torch.zeros_like(wd)
    result = fn(x, gamma, wg, wu, zero_down, 1e-6)
    torch.testing.assert_close(result, x, rtol=0, atol=0)
    dx = torch.autograd.grad(result.sum(), x)[0]
    torch.testing.assert_close(dx, torch.ones_like(x))
    print("练习 C：Pre-RMSNorm 前向、全部梯度、残差恒等路径均通过")

def pre_rms_ffn_ex(x, gamma, w_gate, w_up, w_down, eps=1e-6):
    raise NotImplementedError("请实现 Pre-RMSNorm FFN 残差")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    check_exercise_c(pre_rms_ffn_ex)
else:
    print("练习C未启用：请完成函数后再打开检查")

练习C未启用：请完成函数后再打开检查


### 不运行程序的验收题

1. 为什么常量输入的 LayerNorm 与 RMSNorm 输出不一样？
2. `eps` 放在平方根内和外是不是同一个函数？输入非常小时区别会怎样？
3. 3 个矩阵都用中间宽度 `4D`，是否与普通 FFN 等参数？
4. 为什么本课四组 FFN 等参数，但整个模型仍差 240 个参数？
5. SwiGLU 能否自己让后一个 token 读取前一个 token？
6. 归一化与 FFN 换了之后，旧的 KV cache 为什么不能继续复用？
7. 四组中最低验证损失差 0.006，能否推断真实大模型中也存在相同排名？为什么？

建议把答案写在自己的笔记里；没有必要为了完成课数而跳过这些区别。

---
## 10. 参考答案（完成练习后再读）
参考答案另取函数名，不覆盖你的实现。这里自动运行的是参考答案检查，不代表你的练习已经完成。

### A 的参考实现

In [28]:
def rms_norm_solution(x, weight, eps=1e-6):
    # 前置条件：合法同设备浮点张量，weight=[D]，eps>0。
    # FP64 保留 FP64；只有 FP16/BF16/FP32 才使用 FP32 计算统计量。
    dtype = torch.float64 if x.dtype == torch.float64 else torch.float32
    work = x.to(dtype)
    scale = torch.rsqrt(work.square().mean(dim=-1, keepdim=True) + eps)
    return (work * scale * weight.to(dtype)).to(x.dtype)

check_exercise_a(rms_norm_solution)

练习 A：前向、输入/gamma 梯度、零向量与 FP16 溢出案例均通过


### B 的参考实现

In [29]:
def swiglu_solution(x, w_gate, w_up, w_down):
    gate_input = F.linear(x, w_gate)
    gate = gate_input * torch.sigmoid(gate_input)
    value = F.linear(x, w_up)
    return F.linear(gate * value, w_down)

check_exercise_b(swiglu_solution)

练习 B：三矩阵公式、所有梯度和三个零分支案例均通过


### C 的参考实现

In [30]:
def pre_rms_ffn_solution(x, gamma, w_gate, w_up, w_down, eps=1e-6):
    dtype = torch.float64 if x.dtype == torch.float64 else torch.float32
    work = x.to(dtype)
    normalized = (work * torch.rsqrt(work.square().mean(-1, keepdim=True) + eps)
                  * gamma.to(dtype)).to(x.dtype)
    gate = F.silu(F.linear(normalized, w_gate))
    branch = F.linear(gate * F.linear(normalized, w_up), w_down)
    return x + branch  # 不是 normalized + branch，也不能 detach 原始 x。

check_exercise_c(pre_rms_ffn_solution)

练习 C：Pre-RMSNorm 前向、全部梯度、残差恒等路径均通过


## 11. 常见错误与排查

| 现象 | 优先检查 |
|---|---|
| RMSNorm 输出与官方不同 | eps、是否减均值、归一化轴、gamma、dtype |
| 输出正确但训练不动 | `.detach()`、转成 Python 数字、遗漏参数注册或优化器 |
| 大数输入变成全零 | 是否先在 FP16 里平方溢出，再转换到 FP32 |
| SwiGLU 形状正确但值不对 | SiLU 是否放在正确分支，`*` 是否误写成 `+` |
| 声称等参数但计数不等 | 两个还是三个矩阵、有无偏置、宽度取整、norm 参数 |
| 缓存与完整前向不同 | norm/FFN 是否逐位置、eval、RoPE 偏移、缓存是否来自旧参数 |
| 老检查点加载报错 | 架构已变化，不能靠忽略键把随机参数当成已训练权重 |

导入同名 `.py` 不会训练或生成文件；运行它才会执行入口。Notebook 没有使用 `__file__`。

## 12. 脚本使用与下一课

```bash
python lesson_21_rmsnorm_and_swiglu.py
python lesson_21_rmsnorm_and_swiglu.py --checks-only
python lesson_21_rmsnorm_and_swiglu.py --single --device auto
python lesson_21_rmsnorm_and_swiglu.py --checkpoint lesson_21_outputs/rms_swiglu.pt
```

`--single` 只训练一组，`--epochs 10` 可缩短机制实验；修改这些设置后，不应期待默认报告里的数值。只加载自己的或可信的检查点。

**下一课：GQA 与 MQA。** 比较 Q 头数和 KV 头数，推导 KV cache 大小，验证共享 K/V 的实现和分组关系。届时继续区分数学正确性、模型训练效果和实测性能。

### 原始资料

- [RMSNorm 论文](https://arxiv.org/abs/1910.07467)：为何移除重中心化步骤。
- [GLU Variants Improve Transformer](https://arxiv.org/html/2002.05202v1)：门控 FFN 与中间宽度预算。
- [PyTorch RMSNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.RMSNorm.html)与 [LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)：计算定义与参数。
- [PyTorch SiLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.SiLU.html)：激活函数定义。
- [Meta Llama 3 公开源码](https://github.com/meta-llama/llama3/blob/main/llama/model.py)：将论文概念与实际代码名称对应，勿假定本课检查点兼容。

本讲义代码为独立教学实现；不复制或下载公开模型权重。文献结论的任务与规模不等于本课实验条件。